# Preprocessing of GLAMOS point mass balance data (RGI 11, Switzerland)

This notebook builds the Swiss stake (point mass balance, PMB) dataset used by MassBalanceMachine, from the GLAMOS stake files.

**Pipeline**
1. Convert the GLAMOS `.dat` files to CSV and assemble the annual and winter measurements.
2. Assign RGI IDs (with hand-curated corrections), keep years after 1950 and convert to m w.e.
3. Merge stakes closer than 10 m (same glacier, year and period).
4. Correct or remove stakes whose elevation differs from the yearly GLAMOS DEM by more than 400 m.
5. Sample OGGM topography and drop stakes outside the RGI outline; then sample the SGI 2020 topography and SVF.
6. Re-assign stake IDs by elevation.
7. Clean measurement months and drop stakes with too little data.
8. Save `CH_wgms_dataset_all.csv` to `cfg.dataPath/WGMS/Switzerland/csv/`.

Intermediate files are written to `cfg.dataPath/GLAMOS/point/csv/`.

## Setting up:

In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), '../../../../')) # Add root of repo to import MBM

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import massbalancemachine as mbm

# Swiss paths (path_PMB_GLAMOS_csv, path_GLAMOS_topo, path_SGI_topo, path_PMB_WGMS_csv, ...)
from papers.Transformer_TF.PMB_sources.RGI_11_Switzerland.scripts.config_CH import *

# GLAMOS stake processing and QC (Swiss copies of the shared helpers)
from papers.Transformer_TF.PMB_sources.RGI_11_Switzerland.scripts.glamos import (
    process_pmb_dat_files, process_annual_stake_data, process_winter_stake_data,
    assemble_all_stake_data, add_rgi_ids_to_df, check_multiple_rgi_ids,
    clean_rgi_ids, remove_close_points, reconcile_points_by_year,
    merge_pmb_with_oggm_data, merge_pmb_with_sgi_data,
    rename_stakes_by_elevation, check_point_ids_contain_glacier)
from papers.Transformer_TF.PMB_sources.RGI_11_Switzerland.scripts.geo_data import create_sgi_topo_masks
from papers.Transformer_TF.PMB_sources.RGI_11_Switzerland.scripts.oggm import initialize_oggm_glacier_directories, export_oggm_grids

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Call config file (data paths, seed, etc.)
cfg = mbm.SwitzerlandConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

## Transform .dat files to .csv:

Transform the annual and winter PMB `.dat` files in `GLAMOS/point/point_raw/` to CSV.

In [ ]:
process_pmb_dat_files(cfg)

## Assemble measurement periods:
### Annual measurements:
Process the annual measurements and put all stakes into one CSV file.

In [ ]:
df_annual_raw = process_annual_stake_data(cfg.dataPath + path_PMB_GLAMOS_csv_a)
df_annual_raw.head(2)

### Winter measurements:
For each point in the annual measurements, take the winter measurement that is closest.

In [ ]:
process_winter_stake_data(df_annual_raw, cfg.dataPath + path_PMB_GLAMOS_csv_w,
                          cfg.dataPath + path_PMB_GLAMOS_csv_w_clean)

### Assemble both periods:

In [ ]:
df_all_raw = assemble_all_stake_data(
    df_annual_raw, cfg.dataPath + path_PMB_GLAMOS_csv_w_clean,
    cfg.dataPath + path_PMB_GLAMOS_csv)

# Plot: Number of measurements per year
df_measurements_per_year = df_all_raw.groupby(['YEAR',
                                               'PERIOD']).size().unstack()
df_measurements_per_year.plot(kind='bar',
                              stacked=True,
                              figsize=(20, 5),
                              color=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER])
plt.title('Number of measurements per year for all glaciers')
plt.ylabel('Number of Measurements')
plt.xlabel('Year')
plt.legend(title='Period')
plt.tight_layout()
plt.show()

## Add RGI IDs:
Each stake gets the RGI v6 ID of the outline it falls in. Glaciers that end up with several RGI IDs are corrected by hand (`clean_rgi_ids`).

In [ ]:
df_pmb = add_rgi_ids_to_df(df_all_raw, cfg.dataPath + path_rgi_outlines)

rgiids6 = df_pmb[['GLACIER', 'RGIId']].drop_duplicates()
if check_multiple_rgi_ids(rgiids6):
    print(
        "-- Alert: The following glaciers have more than one RGIId. Cleaning up."
    )
    df_pmb_clean = clean_rgi_ids(df_pmb.copy())
    df_pmb_clean.reset_index(drop=True, inplace=True)

    rgiids6_clean = df_pmb_clean[['GLACIER', 'RGIId']].drop_duplicates()
    if check_multiple_rgi_ids(rgiids6_clean):
        print("-- Error: Some glaciers still have more than one RGIId.")
    else:
        print("-- All glaciers are correctly associated with a single RGIId.")
else:
    print("-- All glaciers are correctly associated with a single RGIId.")
    df_pmb_clean = df_pmb

## Cut from 1951:
Keep years after 1950 (start of ERA5-Land), convert from mm w.e. to m w.e., and merge Clariden L and U into one glacier.

In [ ]:
# Filter to start of MS data (1951) or ERA5-Land data (1950):
df_pmb_50s = df_pmb_clean[df_pmb_clean.YEAR > 1950].sort_values(
    by=['GLACIER', 'YEAR'], ascending=[True, True])

# Change from mm w.e. to m w.e.
df_pmb_50s['POINT_BALANCE'] = df_pmb_50s['POINT_BALANCE'] / 1000

# merge ClaridenL and ClaridenU into one glacier:
df_pmb_50s.loc[df_pmb_50s.GLACIER == 'claridenU', 'GLACIER'] = 'clariden'
df_pmb_50s.loc[df_pmb_50s.GLACIER == 'claridenL', 'GLACIER'] = 'clariden'

print('Number of winter and annual samples:', len(df_pmb_50s))
print('Number of annual samples:',
      len(df_pmb_50s[df_pmb_50s.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_50s[df_pmb_50s.PERIOD == 'winter']))

# Number of measurements per year:
fig, axs = plt.subplots(2, 1, figsize=(20, 15))
ax = axs.flatten()[0]
df_pmb_50s.groupby(['YEAR', 'PERIOD']).size().unstack().plot(
    kind='bar', stacked=True, color=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER], ax=ax)
ax.set_title('Number of measurements per year for all glaciers')

ax = axs.flatten()[1]
num_gl = df_pmb_50s.groupby(['GLACIER']).size().sort_values()
num_gl.plot(kind='bar', ax=ax)
ax.set_title('Number of total measurements per glacier since 1951')
plt.tight_layout()

### Merge stakes that are close: 
Especially with winter probes, a lot of measurements were done at the same place in the raw data and this leads to noise. We merge the stakes that are very close and keep the mean of the measurement.


In [ ]:
df_pmb_50s_clean_pts = pd.DataFrame()
for gl in tqdm(df_pmb_50s.GLACIER.unique(), desc='Merging stakes'):
    print(f'-- {gl.capitalize()}:')
    df_gl = df_pmb_50s[df_pmb_50s.GLACIER == gl]
    df_gl_cleaned = remove_close_points(df_gl)
    df_pmb_50s_clean_pts = pd.concat([df_pmb_50s_clean_pts, df_gl_cleaned])
df_pmb_50s_clean_pts.drop(['x', 'y'], axis=1, inplace=True)

### Correct for wrong elevations:
Some stakes are stored with the wrong elevation. They are compared to the yearly GLAMOS DEMs: on Aletsch, elevations that differ by more than 400 m are replaced by the DEM value; on all other glaciers these stakes are removed.

In [ ]:
path_xr_grids = os.path.join(cfg.dataPath, path_GLAMOS_topo,
                             "xr_masked_grids/")
print("Initial number of rows:", len(df_pmb_50s_clean_pts))

df_pmb_50s_clean_elv, df_mismatch, summary = reconcile_points_by_year(
    df=df_pmb_50s_clean_pts,
    path_xr_grids=path_xr_grids,
    var_name="masked_elev",
    lon_name="lon",
    lat_name="lat",
    year_col="YEAR",
    glacier_col="GLACIER",
    point_elev_col="POINT_ELEVATION",
    threshold=400.0,
    file_pattern="{glacier}_{year}.zarr",
    replace_glaciers={"aletsch"},  # replace for Aletsch, drop for others
    strict=False,
    verbose=True,  # prints counts per glacier
)
df_pmb_50s_clean_elv.reset_index(drop=True, inplace=True)
print("Final number of rows:", len(df_pmb_50s_clean_elv))

# Save mismatches to CSV (for inspection)
out_csv = os.path.join(cfg.dataPath, path_PMB_GLAMOS_csv,
                       "GLAMOS_elev_mismatch.csv")
df_mismatch.sort_values(by="elev_diff", ascending=False, inplace=True)
df_mismatch.to_csv(out_csv, index=False)
print("Saved mismatches to:", out_csv)

# Save intermediate output
print(f'Saving intermediate output df_pmb_50s.csv to {path_PMB_GLAMOS_csv}')
df_pmb_50s_clean_elv.to_csv(os.path.join(cfg.dataPath, path_PMB_GLAMOS_csv,
                                         'df_pmb_50s.csv'),
                            index=False)

In [ ]:
print('Number of winter and annual samples:', len(df_pmb_50s_clean_elv))
print('Number of annual samples:',
      len(df_pmb_50s_clean_elv[df_pmb_50s_clean_elv.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_50s_clean_elv[df_pmb_50s_clean_elv.PERIOD == 'winter']))
# Unique glaciers, sorted
glacier_list = sorted(df_pmb_50s_clean_elv.GLACIER.unique())
print(f"Number of glaciers: {len(glacier_list)}")
print(f"Glaciers: {glacier_list}")

## Add topographical information from OGGM & SGI:

### OGGM data:
Download the OGGM glacier directories of region 11 and sample `aspect`, `slope` and `topo` at each stake. Stakes outside their RGI outline are dropped.

In [ ]:
# Restart point: read the output of the elevation check
df_pmb_50s_clean = pd.read_csv(cfg.dataPath + path_PMB_GLAMOS_csv +
                               'df_pmb_50s.csv')

gdirs, rgidf = initialize_oggm_glacier_directories(
    cfg,
    rgi_region="11",
    rgi_version="62",
    base_url=
    "https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L1-L2_files/2025.6/elev_bands_w_data/",
    log_level='WARNING',
    task_list=None,
)

export_oggm_grids(cfg, gdirs)

df_pmb_topo = merge_pmb_with_oggm_data(
    df_pmb=df_pmb_50s_clean,
    gdirs=gdirs,
    rgi_region="11",
    rgi_version="62",
)

In [ ]:
# restrict to within glacier shape
df_pmb_topo = df_pmb_topo[df_pmb_topo['within_glacier_shape']]
df_pmb_topo = df_pmb_topo.drop(columns=['within_glacier_shape'])

print('Number of winter and annual samples:', len(df_pmb_topo))
print('Number of annual samples:',
      len(df_pmb_topo[df_pmb_topo.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_topo[df_pmb_topo.PERIOD == 'winter']))
# Unique glaciers, sorted
glacier_list = sorted(df_pmb_topo.GLACIER.unique())
print(f"Number of glaciers: {len(glacier_list)}")
print(f"Glaciers: {glacier_list}")

In [ ]:
print(f'Saving intermediate output df_pmb_oggm_intermediate.csv to {path_PMB_GLAMOS_csv}')
df_pmb_topo.to_csv(os.path.join(cfg.dataPath, path_PMB_GLAMOS_csv,
                                'df_pmb_oggm_intermediate.csv'),
                   index=False)

### SGI data:
Build the masked SGI 2020 grids per glacier (aspect, slope, elevation and SVF), then sample them at each stake (`aspect_sgi`, `slope_sgi`, `topo_sgi`, `svf`, `asvf`, `opns`). Stakes outside the SGI mask are dropped.

In [ ]:
# Restart point: read the output of the OGGM step
df_pmb_topo = pd.read_csv(
    os.path.join(cfg.dataPath, path_PMB_GLAMOS_csv,
                 'df_pmb_oggm_intermediate.csv'))

In [ ]:
# First create the masked topographical arrays per glacier:
glacier_list = sorted(df_pmb_topo.GLACIER.unique())
create_sgi_topo_masks(cfg,
                      glacier_list,
                      type='glacier_name',
                      path_save=os.path.join(cfg.dataPath, path_SGI_topo,
                                             'xr_masked_grids/'),
                      path_xr_svf=os.path.join(
                          cfg.dataPath, "GLAMOS/topo/SGI2020/svf_nc_latlon/"))

In [ ]:
path_masked_grids = os.path.join(cfg.dataPath, path_SGI_topo,
                                 'xr_masked_grids/')

# Merge PMB with SGI data
df_pmb_sgi = merge_pmb_with_sgi_data(
    df_pmb_topo,  # cleaned PMB DataFrame
    path_masked_grids,  # path to SGI grids
    voi=[
        "masked_aspect", "masked_slope", "masked_elev", "masked_svf",
        "masked_asvf", "masked_opns"
    ])

df_pmb_sgi.rename(columns={
    'masked_svf': 'svf',
    'masked_asvf': 'asvf',
    'masked_opns': 'opns',
},
                  inplace=True)

# Drop points that have no intersection with SGI mask: (have NaN values)
df_pmb_sgi = df_pmb_sgi.dropna()
df_pmb_sgi.head(2)

In [ ]:
# Count and display the number of samples
print(f"Total number of winter and annual samples: {len(df_pmb_sgi)}")

# Count occurrences of 'PERIOD' values
period_counts = df_pmb_sgi['PERIOD'].value_counts()
print(f"Number of annual samples: {period_counts.get('annual', 0)}")
print(f"Number of winter samples: {period_counts.get('winter', 0)}")

# Unique years, sorted
unique_years = np.sort(df_pmb_sgi.YEAR.unique())
print(f"Unique years: {unique_years}")

# Unique glaciers, sorted
glacier_list = sorted(df_pmb_sgi.GLACIER.unique())
print(f"Number of glaciers: {len(glacier_list)}")
print(f"Glaciers: {glacier_list}")

## Give new stake IDs:
Stake IDs are re-assigned per glacier as `<GLACIER>_<rank>`, where the rank is the stake's mean elevation (0 = lowest). This is needed because the raw stake IDs are not unique across glaciers. Taelliboden (one measurement) and Plaine Morte (large outliers) are dropped first.

In [ ]:
# drop taelliboden (only one measurement)
df_pmb_sgi = df_pmb_sgi[df_pmb_sgi.GLACIER != 'taelliboden']

# drop plainemorte (big outliers)
df_pmb_sgi = df_pmb_sgi[df_pmb_sgi.GLACIER != 'plainemorte']

# Rename POINT_IDs to <GLACIER>_<rank>, ranked by elevation
df_pmb_sgi = rename_stakes_by_elevation(df_pmb_sgi)

# Check that every POINT_ID contains its glacier name
check_point_ids_contain_glacier(df_pmb_sgi)

print('Number of winter and annual samples:', len(df_pmb_sgi))
print('Number of annual samples:',
      len(df_pmb_sgi[df_pmb_sgi.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_sgi[df_pmb_sgi.PERIOD == 'winter']))

# Histogram of mass balance
df_pmb_sgi['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

## Final cleaning:
1. Remove measurements that start in July, and annual measurements that end in June or November.
2. Relabel annual measurements ending in May as winter, and winter measurements ending in August as annual.
3. Remove stakes with too much missing data.
4. Save `CH_wgms_dataset_all.csv`.

In [ ]:
# Months (MM) of start and end of each measurement
df_pmb_sgi['MONTH_START'] = [str(date)[4:6] for date in df_pmb_sgi.FROM_DATE]
df_pmb_sgi['MONTH_END'] = [str(date)[4:6] for date in df_pmb_sgi.TO_DATE]

# drop rows where month_start is '07'
df_pmb_sgi = df_pmb_sgi[df_pmb_sgi['MONTH_START'] != '07']

# drop annual measurements ending in June or November
df_pmb_sgi = df_pmb_sgi.loc[~((df_pmb_sgi['MONTH_END'] == '06') &
                              (df_pmb_sgi['PERIOD'] == 'annual'))]

df_pmb_sgi = df_pmb_sgi.loc[~((df_pmb_sgi['MONTH_END'] == '11') &
                              (df_pmb_sgi['PERIOD'] == 'annual'))]

# Rows where month_end is '05' and period is 'annual', rename period to 'winter'
df_pmb_sgi.loc[(df_pmb_sgi['MONTH_END'] == '05') &
               (df_pmb_sgi['PERIOD'] == 'annual'), 'PERIOD'] = 'winter'

# Rows where month_end is '08' and period is 'winter', rename period to 'annual'
df_pmb_sgi.loc[(df_pmb_sgi['MONTH_END'] == '08') &
               (df_pmb_sgi['PERIOD'] == 'winter'), 'PERIOD'] = 'annual'

POINT_ID_to_drop = [
    'schwarzberg_12', 'schwarzberg_13', 'schwarzberg_6', 'schwarzberg_5',
    'plattalva_7', 'plattalva_11', 'plattalva_10', 'plattalva_1'
]

# remove points with too much missing data
df_pmb_sgi = df_pmb_sgi[~df_pmb_sgi['POINT_ID'].isin(POINT_ID_to_drop)]

# Save to csv (WGMS/Switzerland/csv/, where data_loader.py reads it):
os.makedirs(cfg.dataPath + path_PMB_WGMS_csv, exist_ok=True)
df_pmb_sgi.to_csv(cfg.dataPath + path_PMB_WGMS_csv +
                  'CH_wgms_dataset_all.csv',
                  index=False)

print('Number of winter and annual samples:', len(df_pmb_sgi))
print('Number of glaciers:', df_pmb_sgi.GLACIER.nunique())